# 01 — Minimal end-to-end loop (TF-IDF + LR)

In [1]:
%load_ext autoreload
%autoreload 2

import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))  # so `import src...` works from notebooks/

import numpy as np
import pandas as pd

from src import config
from src.utils import set_seed, save_fig
set_seed()  # fix all RNGs -- reproducibility

Goal: light up the whole chain — load → split → **one** feature → **one** classifier → Macro-F1 + confusion. Everything else is filling this in.

In [2]:
from src import data, features, modeling

clean = data.load_or_build_clean()
splits = data.load_or_build_splits(clean)
y = clean[config.LABEL_COL].values

In [3]:
# TF-IDF only
X_tfidf, vec = features.build_tfidf(clean.iloc[splits['train']]['text'], clean['text'])
Xtr, ytr = X_tfidf[splits['train']], y[splits['train']]
Xval, yval = X_tfidf[splits['val']], y[splits['val']]

In [4]:
res = modeling.train_and_evaluate('logreg', Xtr, ytr, Xval, yval)
print('Macro-F1:', res.macro_f1)
res.per_class

Macro-F1: 0.7418533834362195


{'chatgpt': {'precision': 0.869410569105691,
  'recall': 0.897691500524659,
  'f1': 0.8833247289623128},
 'cohere': {'precision': 0.622680412371134,
  'recall': 0.6351209253417456,
  'f1': 0.6288391462779802},
 'cohere-chat': {'precision': 0.6494178525226391,
  'recall': 0.5284210526315789,
  'f1': 0.5827045850261172},
 'gpt2': {'precision': 0.7205177372962608,
  'recall': 0.7973474801061008,
  'f1': 0.7569881641903803},
 'gpt3': {'precision': 0.7215645908389089,
  'recall': 0.7398416886543535,
  'f1': 0.7305888483585201},
 'gpt4': {'precision': 0.9235171696149844,
  'recall': 0.9293193717277487,
  'f1': 0.9264091858037579},
 'human': {'precision': 0.7520585544373285,
  'recall': 0.8657187993680885,
  'f1': 0.8048959608323133},
 'llama-chat': {'precision': 0.8572151898734177,
  'recall': 0.8948202959830867,
  'f1': 0.8756141711921386},
 'mistral': {'precision': 0.6402918069584737,
  'recall': 0.6037037037037037,
  'f1': 0.6214596949891068},
 'mistral-chat': {'precision': 0.763733333333

Once this prints a sane Macro-F1 and confusion matrix, the skeleton works. Move on to fleshing out `train_and_evaluate` metrics, then features.